# Modeling Human Activity States Using Hidden Markov Models

Hidden states **Z** = {standing, walking, jumping, still}. Observations **X** = feature vectors (time + frequency domain) from accelerometer and gyroscope windows. The HMM is implemented from scratch with numpy.

**Expected data layout** (relative to the repo root)

```
data/raw/        training recordings
data/test/       unseen recordings (a separate session: another day, place or phone position), never used for training
data/processed/  resampled recordings written by this notebook
```

Each recording is named `<name>_<activity>_<nn>`, e.g. `kalisa10_walking_03`, and is either
* a **folder** holding Sensor Logger's `Accelerometer.csv` and `Gyroscope.csv`, or
* a single **CSV** with columns `timestamp, ax, ay, az, gx, gy, gz`.

**Pipeline:** load and resample, window, features, standardize + PCA, HMM (supervised counting, Viterbi), evaluation, plots.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path(os.environ.get('HMM_DATA_DIR', '../data'))
RAW_DIR, TEST_DIR, PROC_DIR = DATA_DIR / 'raw', DATA_DIR / 'test', DATA_DIR / 'processed'
FIG_DIR = Path(os.environ.get('HMM_FIG_DIR', '../report/figures'))
FIG_DIR.mkdir(parents=True, exist_ok=True)

ACTIVITIES = ['standing', 'walking', 'jumping', 'still']
K = len(ACTIVITIES)
CHANNELS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']

TARGET_FS = 50      # Hz, common sampling rate for every member
TRIM_SEC = 1.0      # drop the start/end of each recording (tapping the phone)
TRIM_END_OVERRIDE = {'kalisa10_standing_01': 3.5}   # phone was moved during the last seconds of this clip
WIN_SEC = 2.0       # window length
OVERLAP = 0.5       # window overlap
N_PCA = 8           # PCA components fed to the HMM
SESSION_LEN = 5     # recordings stitched into one session (gives real transitions)
N_SESSIONS = 40     # number of stitched sessions drawn per pool
SEED = 42
rng = np.random.default_rng(SEED)

## 1. Load and harmonize sampling rates

In [ ]:
def to_seconds(t):
    """Seconds since the first sample; handles ns / ms / s epoch timestamps."""
    t = np.asarray(t, dtype=float)
    scale = 1e9 if t[0] > 1e17 else 1e3 if t[0] > 1e11 else 1.0
    return (t - t[0]) / scale


def parse_name(path):
    member, activity, _ = Path(path).stem.rsplit('_', 2)
    assert activity in ACTIVITIES, f'unknown activity in {path}'
    return member, activity


def load_recording(path):
    """Return (DataFrame of the 6 channels resampled to TARGET_FS, native sampling rate)."""
    path = Path(path)
    if path.is_dir():                                   # Sensor Logger export
        # TotalAcceleration keeps gravity, so the mean features encode phone orientation
        # (flat on a table vs upright at the waist); Accelerometer.csv has gravity removed.
        acc_file = 'TotalAcceleration.csv' if (path / 'TotalAcceleration.csv').exists() else 'Accelerometer.csv'
        acc = pd.read_csv(path / acc_file)
        gyr = pd.read_csv(path / 'Gyroscope.csv')
        ta = acc['seconds_elapsed'].to_numpy()
        tg = gyr['seconds_elapsed'].to_numpy()
        series = {**{f'a{c}': (ta, acc[c].to_numpy()) for c in 'xyz'},
                  **{f'g{c}': (tg, gyr[c].to_numpy()) for c in 'xyz'}}
    else:                                               # merged CSV
        df = pd.read_csv(path)
        t = to_seconds(df['timestamp'])
        ta = tg = t
        series = {c: (t, df[c].to_numpy()) for c in CHANNELS}
    native_fs = 1.0 / np.median(np.diff(ta))
    start = max(ta[0], tg[0]) + TRIM_SEC
    stop = min(ta[-1], tg[-1]) - TRIM_END_OVERRIDE.get(path.stem, TRIM_SEC)
    grid = np.arange(start, stop, 1.0 / TARGET_FS)
    out = pd.DataFrame({'timestamp': grid - grid[0]})
    for c in CHANNELS:
        out[c] = np.interp(grid, *series[c])
    return out, native_fs


def list_recordings(folder):
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob('*')
                  if (p.is_dir() and (p / 'Accelerometer.csv').exists())
                  or (p.is_file() and p.suffix == '.csv'
                      and p.stem.count('_') >= 2 and p.name not in ('Accelerometer.csv', 'Gyroscope.csv')))


def load_pool(folder, save_to=None):
    recs, rows = [], []
    for p in list_recordings(folder):
        member, activity = parse_name(p)
        df, native_fs = load_recording(p)
        recs.append({'name': p.stem, 'member': member, 'activity': activity, 'data': df})
        rows.append({'name': p.stem, 'member': member, 'activity': activity,
                     'native_fs_hz': round(native_fs, 1), 'seconds': round(len(df) / TARGET_FS, 1)})
        if save_to is not None:
            save_to.mkdir(parents=True, exist_ok=True)
            df.to_csv(save_to / f'{p.stem}.csv', index=False)
    return recs, pd.DataFrame(rows)


train_pool, inv_train = load_pool(RAW_DIR, save_to=PROC_DIR)
test_pool, inv_test = load_pool(TEST_DIR)
print(f'{len(train_pool)} training recordings, {len(test_pool)} unseen recordings')
inv_train.groupby(['member', 'activity']).size().unstack(fill_value=0)

In [ ]:
# Native sampling rate of the recording phone (for the report)
pd.concat([inv_train, inv_test]).groupby('member')['native_fs_hz'].agg(['mean', 'min', 'max']).round(1)

In [ ]:
# Example raw signal per activity (accelerometer magnitude)
fig, axes = plt.subplots(K, 1, figsize=(9, 7), sharex=True)
for ax, act in zip(axes, ACTIVITIES):
    rec = next(r for r in train_pool if r['activity'] == act)
    d = rec['data']
    ax.plot(d['timestamp'], np.linalg.norm(d[['ax', 'ay', 'az']], axis=1), lw=1)
    ax.set_ylabel(act)
axes[-1].set_xlabel('time (s)')
fig.suptitle('Accelerometer magnitude, one recording per activity')
fig.tight_layout(); fig.savefig(FIG_DIR / 'raw_signals.png', dpi=150); plt.show()

## 2. Windowing and feature extraction

Per window (2 s, 50 % overlap) and per channel: mean, variance, standard deviation.
Across channels: signal magnitude area (SMA) and correlation between axes for both sensors.
Frequency domain on the accelerometer and gyroscope magnitudes: dominant frequency, spectral energy, and energy in four frequency bands (FFT components).

In [ ]:
BANDS = [(0.3, 1), (1, 3), (3, 6), (6, 15)]   # Hz


def spectrum_feats(sig, fs):
    sig = sig - sig.mean()
    spec = np.abs(np.fft.rfft(sig)) ** 2 / len(sig)
    freqs = np.fft.rfftfreq(len(sig), 1 / fs)
    dom = freqs[1:][np.argmax(spec[1:])]
    bands = [spec[(freqs >= lo) & (freqs < hi)].sum() for lo, hi in BANDS]
    return [dom, spec[1:].sum(), *bands]


def window_features(w, fs=TARGET_FS):
    acc, gyr = w[:, :3], w[:, 3:]
    f = [*w.mean(0), *w.var(0), *w.std(0)]
    for s in (acc, gyr):
        f.append(np.abs(s).sum() / len(s))                       # SMA
        with np.errstate(invalid='ignore', divide='ignore'):
            c = np.corrcoef(s.T)
        f += [c[0, 1], c[0, 2], c[1, 2]]                         # axis correlations
        f += spectrum_feats(np.linalg.norm(s, axis=1), fs)       # frequency domain
    return np.nan_to_num(np.array(f))                            # flat signal -> corr NaN -> 0


def feature_names():
    n = [f'{s}_{c}' for s in ('mean', 'var', 'std') for c in CHANNELS]
    for p in ('acc', 'gyr'):
        n += [f'{p}_sma', f'{p}_corr_xy', f'{p}_corr_xz', f'{p}_corr_yz', f'{p}_domfreq', f'{p}_energy']
        n += [f'{p}_band_{lo}-{hi}Hz' for lo, hi in BANDS]
    return n


def featurize(rec):
    x = rec['data'][CHANNELS].to_numpy()
    size, step = int(WIN_SEC * TARGET_FS), int(WIN_SEC * TARGET_FS * (1 - OVERLAP))
    X = np.array([window_features(x[i:i + size]) for i in range(0, len(x) - size + 1, step)])
    return X, np.full(len(X), ACTIVITIES.index(rec['activity']))


for r in train_pool + test_pool:
    r['X'], r['y'] = featurize(r)

FEATS = feature_names()
print(len(FEATS), 'features per window;', sum(len(r['y']) for r in train_pool), 'training windows,',
      sum(len(r['y']) for r in test_pool), 'unseen windows')

## 3. Train / unseen split

If `data/test/` is empty the notebook falls back to holding out 25 % of the recordings of each activity so it still runs, but **the assignment asks for genuinely new recordings** (a separate session: another day, another place or a different phone position) in `data/test/`.

In [ ]:
if test_pool:
    train_recs, test_recs = train_pool, test_pool
    print('Using data/test/ as unseen data')
else:
    print('WARNING: data/test/ is empty, holding out 25% of recordings per activity')
    train_recs, test_recs = [], []
    for act in ACTIVITIES:
        pool = [r for r in train_pool if r['activity'] == act]
        idx = rng.permutation(len(pool))
        n_test = max(1, round(0.25 * len(pool)))
        test_recs += [pool[i] for i in idx[:n_test]]
        train_recs += [pool[i] for i in idx[n_test:]]


def make_sessions(recs, n_sessions, length):
    """Stitch random recordings into longer sessions so the sequence contains activity changes."""
    sessions = []
    for _ in range(n_sessions):
        pick = rng.choice(len(recs), size=min(length, len(recs)), replace=False)
        sessions.append((np.vstack([recs[i]['X'] for i in pick]),
                         np.concatenate([recs[i]['y'] for i in pick])))
    return sessions


train_sessions = make_sessions(train_recs, N_SESSIONS, SESSION_LEN)
test_sessions = make_sessions(test_recs, max(10, N_SESSIONS // 2), SESSION_LEN)
print(len(train_recs), 'train recordings ->', len(train_sessions), 'sessions;',
      len(test_recs), 'unseen recordings ->', len(test_sessions), 'sessions')

## 4. Standardize and reduce (fitted on training windows only)

In [ ]:
Xtr_all = np.vstack([r['X'] for r in train_recs])
mu, sd = Xtr_all.mean(0), Xtr_all.std(0) + 1e-9
_, S, Vt = np.linalg.svd((Xtr_all - mu) / sd, full_matrices=False)
W = Vt[:N_PCA].T
print(f'PCA keeps {100 * (S[:N_PCA] ** 2).sum() / (S ** 2).sum():.1f}% of the variance')


def transform(X):
    return ((X - mu) / sd) @ W


train_sessions = [(transform(X), y) for X, y in train_sessions]
test_sessions = [(transform(X), y) for X, y in test_sessions]

## 5. Hidden Markov Model from scratch

* **Z**: the 4 activities, **X**: PCA feature vectors.
* **π**: how often each activity starts a session. **A**: transition counts between consecutive windows. **B**: one multivariate Gaussian per activity.
* Labels are known, so parameters are estimated by **counting** (maximum likelihood, with additive smoothing) and Gaussian moments, with no EM needed.
* **Viterbi** (log-space) decodes the single most likely activity sequence, which is what the per-activity sensitivity and specificity are computed from. **Forward-Backward** gives per-window posteriors (confidence).

In [ ]:
def logsumexp(a, axis=None):
    m = np.max(a, axis=axis, keepdims=True)
    out = m + np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True))
    return out if axis is None else np.squeeze(out, axis=axis)


class GaussianHMM:
    def __init__(self, n_states, smooth=1.0, reg=1e-2):
        self.K, self.smooth, self.reg = n_states, smooth, reg

    def fit(self, sessions):
        D = sessions[0][0].shape[1]
        pi = np.full(self.K, self.smooth)
        A = np.full((self.K, self.K), self.smooth)
        for X, y in sessions:
            pi[y[0]] += 1
            np.add.at(A, (y[:-1], y[1:]), 1)
        self.pi, self.A = pi / pi.sum(), A / A.sum(1, keepdims=True)
        Xall, yall = np.vstack([s[0] for s in sessions]), np.concatenate([s[1] for s in sessions])
        self.mu = np.array([Xall[yall == k].mean(0) for k in range(self.K)])
        self.cov = np.array([np.cov(Xall[yall == k].T) + self.reg * np.eye(D) for k in range(self.K)])
        return self

    def log_emission(self, X):
        T, D = X.shape
        out = np.empty((T, self.K))
        for k in range(self.K):
            L = np.linalg.cholesky(self.cov[k])
            z = np.linalg.solve(L, (X - self.mu[k]).T)
            out[:, k] = -0.5 * (z ** 2).sum(0) - np.log(np.diag(L)).sum() - 0.5 * D * np.log(2 * np.pi)
        return out

    def viterbi(self, X):
        logB, logA = self.log_emission(X), np.log(self.A)
        T = len(X)
        delta, psi = np.empty((T, self.K)), np.zeros((T, self.K), dtype=int)
        delta[0] = np.log(self.pi) + logB[0]
        for t in range(1, T):
            scores = delta[t - 1][:, None] + logA
            psi[t] = scores.argmax(0)
            delta[t] = scores.max(0) + logB[t]
        path = np.empty(T, dtype=int)
        path[-1] = delta[-1].argmax()
        for t in range(T - 2, -1, -1):
            path[t] = psi[t + 1, path[t + 1]]
        return path

    def posteriors(self, X):
        logB, logA = self.log_emission(X), np.log(self.A)
        T = len(X)
        la, lb = np.empty((T, self.K)), np.zeros((T, self.K))
        la[0] = np.log(self.pi) + logB[0]
        for t in range(1, T):
            la[t] = logsumexp(la[t - 1][:, None] + logA, axis=0) + logB[t]
        for t in range(T - 2, -1, -1):
            lb[t] = logsumexp(logA + logB[t + 1] + lb[t + 1], axis=1)
        g = la + lb
        return np.exp(g - logsumexp(g, axis=1)[:, None])


hmm = GaussianHMM(K).fit(train_sessions)
print('pi =', dict(zip(ACTIVITIES, hmm.pi.round(3))))

## 6. Transition matrix heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(hmm.A, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(K), ACTIVITIES, rotation=30); ax.set_yticks(range(K), ACTIVITIES)
for i in range(K):
    for j in range(K):
        ax.text(j, i, f'{hmm.A[i, j]:.2f}', ha='center', va='center',
                color='white' if hmm.A[i, j] > 0.5 else 'black')
ax.set_xlabel('to state'); ax.set_ylabel('from state'); ax.set_title('Transition matrix A')
fig.colorbar(im); fig.tight_layout(); fig.savefig(FIG_DIR / 'transition_matrix.png', dpi=150); plt.show()

## 7. Evaluation on unseen data

Sensitivity, specificity and accuracy are computed **one-vs-rest** per activity from the confusion matrix of Viterbi-decoded windows. A frame-by-frame Gaussian classifier (no transitions) is shown as a baseline to measure what the HMM's transition structure adds.

In [ ]:
def confusion(y_true, y_pred):
    C = np.zeros((K, K), dtype=int)
    np.add.at(C, (y_true, y_pred), 1)
    return C


def metrics_table(C):
    rows, N = [], C.sum()
    for k, act in enumerate(ACTIVITIES):
        TP = C[k, k]; FN = C[k].sum() - TP; FP = C[:, k].sum() - TP; TN = N - TP - FN - FP
        rows.append({'State (Activity)': act, 'Number of Samples': int(C[k].sum()),
                     'Sensitivity': TP / (TP + FN) if TP + FN else np.nan,
                     'Specificity': TN / (TN + FP) if TN + FP else np.nan,
                     'Overall Accuracy': (TP + TN) / N})
    df = pd.DataFrame(rows)
    df.loc[len(df)] = ['ALL (overall accuracy)', int(N), np.nan, np.nan, np.trace(C) / N]
    return df


y_true = np.concatenate([y for _, y in test_sessions])
y_hmm = np.concatenate([hmm.viterbi(X) for X, _ in test_sessions])
y_frame = np.concatenate([hmm.log_emission(X).argmax(1) for X, _ in test_sessions])

C_hmm = confusion(y_true, y_hmm)
table = metrics_table(C_hmm)
table.to_csv(FIG_DIR / 'metrics_table.csv', index=False)
print(f'HMM (Viterbi) accuracy: {np.trace(C_hmm) / C_hmm.sum():.3f}   '
      f'frame-wise Gaussian baseline: {(y_frame == y_true).mean():.3f}')
table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(C_hmm, cmap='Blues')
ax.set_xticks(range(K), ACTIVITIES, rotation=30); ax.set_yticks(range(K), ACTIVITIES)
for i in range(K):
    for j in range(K):
        ax.text(j, i, C_hmm[i, j], ha='center', va='center',
                color='white' if C_hmm[i, j] > C_hmm.max() / 2 else 'black')
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Confusion matrix (unseen data)')
fig.tight_layout(); fig.savefig(FIG_DIR / 'confusion_matrix.png', dpi=150); plt.show()

## 8. Decoded sequences

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharey=True)
for ax, (X, y) in zip(axes, test_sessions[:3]):
    pred = hmm.viterbi(X)
    t = np.arange(len(y)) * WIN_SEC * (1 - OVERLAP)
    ax.step(t, y, where='post', lw=3, alpha=0.4, label='true')
    ax.step(t, pred, where='post', lw=1.5, ls='--', color='tab:red', label='decoded (Viterbi)')
    ax.set_yticks(range(K), ACTIVITIES)
    ax.set_title(f'accuracy {(pred == y).mean():.2f}', fontsize=9, loc='right')
axes[0].legend(loc='upper left', ncol=2); axes[-1].set_xlabel('time (s)')
fig.suptitle('Decoded activity sequences on unseen sessions')
fig.tight_layout(); fig.savefig(FIG_DIR / 'decoded_sequences.png', dpi=150); plt.show()

In [ ]:
# Forward-Backward posteriors: how confident is the model along one session?
X, y = test_sessions[0]
post = hmm.posteriors(X)
fig, ax = plt.subplots(figsize=(10, 3))
ax.stackplot(np.arange(len(y)) * WIN_SEC * (1 - OVERLAP), post.T, labels=ACTIVITIES, alpha=0.85)
ax.set_xlabel('time (s)'); ax.set_ylabel('P(state | all data)'); ax.legend(ncol=4, loc='lower center')
fig.tight_layout(); fig.savefig(FIG_DIR / 'posteriors.png', dpi=150); plt.show()

## 9. Notes for the report

* Which activities have the lowest sensitivity? Look at the off-diagonal cells of the confusion matrix (still vs standing is the usual suspect).
* Compare the HMM and the frame-wise baseline above: the gap is the value of the transition matrix.
* Do the rows of **A** show strong self-transitions (activities persist)? Remember the sessions are stitched from separate recordings, so the between-activity probabilities reflect how they were stitched. Say so in the report, or record continuous sessions with real activity changes.
* Discuss the phone's native sampling rate vs the 50 Hz target (section 1), sensor noise, and improvements (more participants and sessions, more features, more sensors).